In [1]:
%pip install -q --upgrade pageindex

Note: you may need to restart the kernel to use updated packages.


## Step 1: Install PageIndex

In [1]:
import os 
from pageindex import PageIndexClient
import pageindex.utils as utils
from dotenv import load_dotenv

load_dotenv()

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

## Step 2: Set Up the LLM

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load the environment variables from the .env file into your OS environment.
# This MUST happen before initializing the client.
load_dotenv()

# 2. Initialize the client. It will now successfully find GEMINI_API_KEY.
client = genai.Client()

import re

async def call_llm(prompt, model="gemini-2.5-flash", temperature=0):
    response = await client.aio.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
        )
    )
    
    text = response.text.strip()
    # Strip markdown code fences (```json ... ``` or ``` ... ```)
    text = re.sub(r'^```(?:json)?\s*\n', '', text)
    text = re.sub(r'\n```\s*$', '', text)
    return text


## Step 3: Import Libraries & Connect to PageIndex

In [3]:
pdf_path = "2.pdf"
doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print('Document Submitted:', doc_id)

Document Submitted: pi-cmnc1p44m06nt0gqst1ee35yf


## Step 4: Upload Your PDF

In [4]:
import time

print("Waiting for document to be processed...")
while not pi_client.is_retrieval_ready(doc_id):
    print("Still processing... retrying in 10 seconds")
    time.sleep(10)

tree = pi_client.get_tree(doc_id, node_summary=True)['result']
print('Simplified Tree Structure of the Document:')
utils.print_tree(tree)

Waiting for document to be processed...
Still processing... retrying in 10 seconds
Still processing... retrying in 10 seconds
Simplified Tree Structure of the Document:
[{'title': 'Preface', 'node_id': '0000', 'summary': 'The text describes Free Trade Agreements...'},
 {'title': "Summary Table: India's Active FTAs",
  'node_id': '0001',
  'prefix_summary': "# Summary Table: India's Active FTAs\n\n| ...",
  'nodes': [{'title': '1. Why (Purpose / Objective)',
             'node_id': '0002',
             'summary': '## 1. Why (Purpose / Objective)\n\n- To bo...'},
            {'title': '2. When (Timing / Context)',
             'node_id': '0003',
             'summary': '## 2. When (Timing / Context)\n\n- FTAs ar...'},
            {'title': '3. Who (Stakeholders / Signatories)',
             'node_id': '0004',
             'summary': '## 3. Who (Stakeholders / Signatories)\n\n...'},
            {'title': '4. Where (Venue / Scope)',
             'node_id': '0005',
             'summary': '

## Step 5: Wait for Processing & Retrieve the Document Tree

In [5]:
import json

query = "when India-South Korea CEPA agreement became effective?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_llm(search_prompt)
print(tree_search_result)

{
    "thinking": "The question asks for the effective date of the India-South Korea CEPA agreement. I need to look for nodes that mention specific agreements and their operational or effective dates. \n\n- Node '0000' (Preface) explicitly states that the document will list 'bilateral FTAs, Comprehensive Economic Cooperation Agreements (CECA), Comprehensive Economic Partnership Agreements (CEPA), and Trade and Economic Partnership Agreements (TEPA) that India has in force with various countries and regions, noting their operational status and key dates.' This is highly relevant as it promises the exact type of information requested.\n- Node '0001' (Summary Table: India's Active FTAs) lists 'South Korea' under 'Bilateral / Regional FTAs'. While it doesn't provide the date directly, it confirms that the India-South Korea agreement is an active FTA discussed in the document, and its prefix summary also mentions 'key dates'. This node acts as an index or confirmation that the specific agre

## Step 6: Search the Document Tree with an LLM

In [6]:
node_map = utils.create_node_mapping(tree)

# Debug: Check if tree_search_result is valid JSON
print("Raw tree_search_result:")
print(repr(tree_search_result))
print()

try:
    tree_search_result_json = json.loads(tree_search_result)
except json.JSONDecodeError as e:
    print(f"Error parsing JSON: {e}")
    print("LLM response might not be valid JSON. Trying to extract JSON from response...")
    # Try to find JSON in the response
    import re
    json_match = re.search(r'\{.*\}', tree_search_result, re.DOTALL)
    if json_match:
        tree_search_result_json = json.loads(json_match.group())
    else:
        raise ValueError("Could not find valid JSON in LLM response")

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Raw tree_search_result:
'{\n    "thinking": "The question asks for the effective date of the India-South Korea CEPA agreement. I need to look for nodes that mention specific agreements and their operational or effective dates. \\n\\n- Node \'0000\' (Preface) explicitly states that the document will list \'bilateral FTAs, Comprehensive Economic Cooperation Agreements (CECA), Comprehensive Economic Partnership Agreements (CEPA), and Trade and Economic Partnership Agreements (TEPA) that India has in force with various countries and regions, noting their operational status and key dates.\' This is highly relevant as it promises the exact type of information requested.\\n- Node \'0001\' (Summary Table: India\'s Active FTAs) lists \'South Korea\' under \'Bilateral / Regional FTAs\'. While it doesn\'t provide the date directly, it confirms that the India-South Korea agreement is an active FTA discussed in the document, and its prefix summary also mentions \'key dates\'. This node acts as an i

## Step 7: Review the Retrieved Nodes

In [7]:
node_list = tree_search_result_json["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await call_llm(answer_prompt)
utils.print_wrapped(answer)

Retrieved Context:

Free Trade Agreements: FTA:

1. Multilateral regional agreements and
2. Bilateral partnerships with countries across Asia, the Middle East, Africa, and Europe.

These FTAs serve to reduce tariffs, promote trade flows in goods and services, and often include
provisions on investment, mobility, intellectual property etc.

Multilateral &amp; Preferential Trade Agreements (In Force)

- South Asian Free Trade Area (SAFTA) – Among SAARC nations: Afghanistan, Bangladesh, Bhutan, India,
Maldives, Nepal, Pakistan, Sri Lanka.
- Asia-Pacific Trade Agreement (APTA) – Includes Bangladesh, China, South Korea, Sri Lanka, and
others.
- Global System of Trade Preferences (GSTP) – A multilateral preferential regime involving 42
countries, including India.

Bilateral FTAs / CEPAs / CECAs (In Force)

FTAs between India and other countries/regions:

- India–Sri Lanka FTA – Bilateral FTA with Sri Lanka.
- India–Nepal Treaty of Trade – Preferential access between India and Nepal.
- India–

## Step 8: Generate the Final Answer

In [8]:
async def ask(query):
    tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

    search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

    search_result = await call_llm(search_prompt)
    search_result_json = json.loads(search_result)

    node_list = search_result_json["node_list"]
    relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

    answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

    answer = await call_llm(answer_prompt)

    print(f"Query: {query}")
    print(f"\nRelevant Nodes: {node_list}")
    print("\nAnswer:")
    utils.print_wrapped(answer)
    print("-" * 60)

# --- Enter your query below and run this cell ---
user_query = input("Enter your query: ")
await ask(user_query)

Query: when India-South Korea CEPA agreement became effective?

Relevant Nodes: ['0000', '0001']

Answer:
The India-South Korea CEPA agreement became effective since January 1, 2010.
------------------------------------------------------------
